In [1]:
import anthropic
import getpass
import os

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"


In [2]:
response = client.messages.create(  
    model=MODEL,
    max_tokens=256,                  
    messages=[{"role": "user", "content": "What's the weather in Tokyo?"}],  
)
print("No tools:", response.content[0].text)  

No tools: I don't have access to real-time weather data. To check Tokyo's current weather, I'd recommend:

- **Google Weather** - Search "Tokyo weather"
- **Weather.com** - Detailed forecasts
- **Japan Meteorological Agency** (JMA) - Official Japanese weather service
- **Weather apps** - Apple Weather, Weather Underground, etc.

Is there anything else about Tokyo I can help you with?


In [3]:
def get_weather(city: str) -> str:
    return "22°C, partly cloudy"


tools = [{
    "name": "get_weather",
    "description": "Get the current weather for a city",
    "input_schema": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
    },
}]

In [4]:
r1 = client.messages.create(
    model=MODEL,
    max_tokens=256,
    tools=tools,
    messages=[{"role": "user", "content": "What's the weather in Tokyo?"}],
)

In [5]:
r1

Message(id='msg_0112EzynBmamPHX8d7LRSLKG', container=None, content=[ToolUseBlock(id='toolu_01Bg1mcrPjFLi9hAZMDcUBZ6', caller=DirectCaller(type='direct'), input={'city': 'Tokyo'}, name='get_weather', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=565, output_tokens=54, server_tool_use=None, service_tier='standard'))

In [6]:
r1.stop_reason

'tool_use'

In [7]:
r1.content[0]

ToolUseBlock(id='toolu_01Bg1mcrPjFLi9hAZMDcUBZ6', caller=DirectCaller(type='direct'), input={'city': 'Tokyo'}, name='get_weather', type='tool_use')

In [8]:
tool_call = next(b for b in r1.content if b.type == "tool_use")

In [9]:
tool_call

ToolUseBlock(id='toolu_01Bg1mcrPjFLi9hAZMDcUBZ6', caller=DirectCaller(type='direct'), input={'city': 'Tokyo'}, name='get_weather', type='tool_use')

In [10]:
result = get_weather(**tool_call.input)

In [11]:
print(f"Tool called: {tool_call.name}({tool_call.input}) → {result}")

Tool called: get_weather({'city': 'Tokyo'}) → 22°C, partly cloudy


In [12]:
print(tool_call.id)

toolu_01Bg1mcrPjFLi9hAZMDcUBZ6


In [13]:
r2 = client.messages.create(
    model=MODEL,
    max_tokens=256,
    tools=tools,
    messages=[
        {"role": "user", "content": "What's the weather in Tokyo?"},
        {"role": "assistant", "content": r1.content},      
        {"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_call.id, "content": result}]},
    ],
)

In [14]:
print("With tools:", next(b.text for b in r2.content if b.type == "text"))

With tools: The weather in Tokyo is currently **22°C (about 72°F) and partly cloudy**. It's a pleasant day!
